# Employee & Workforce Intelligence System: ML Notebook
AI-Based Employee & Workforce Intelligence System using Machine Learning for smart HR decision-making.

**Pipeline:** Data collection → Cleaning → Feature selection → Model training → Prediction → Segmentation → Anomaly detection → HR insights.

> *Google Colab:* upload `employee_hr_data.csv` (from the `data/` folder) using the file panel, then set `CSV` below.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, IsolationForest
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, roc_auc_score, r2_score, mean_absolute_error, silhouette_score
sns.set_theme(style="whitegrid")

CSV = "../data/employee_hr_data.csv"        # on Colab: "employee_hr_data.csv"
df = pd.read_csv(CSV)
print(df.shape); df.head()

## 1. Data cleaning

In [ ]:
print("Duplicates:", df.duplicated("employee_id").sum())
print("Missing values per column:"); print(df.isna().sum()[df.isna().sum() > 0])

df = df.drop_duplicates("employee_id").reset_index(drop=True)
cat_cols = ["gender","department","job_role","education","overtime","business_travel"]
num_cols = ["age","distance_from_home","job_level","total_working_years","years_at_company",
            "years_since_last_promotion","num_companies_worked","monthly_income","last_salary_hike_pct",
            "job_satisfaction","work_life_balance","environment_satisfaction","job_involvement","manager_rating",
            "training_hours_last_year","trainings_attended","projects_completed","absenteeism_days"]
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
for c in cat_cols: df[c] = df[c].fillna(df[c].mode()[0])
df["attrition_flag"]  = (df.attrition == "Yes").astype(int)
df["promotion_flag"]  = (df.promoted_next_cycle == "Yes").astype(int)
df["training_flag"]   = (df.needs_training == "Yes").astype(int)
df["overtime_flag"]   = (df.overtime == "Yes").astype(int)
print("Clean shape:", df.shape)

## 2. Exploratory analysis

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=df, x="overtime", y="attrition_flag", ax=ax[0]); ax[0].set_title("Attrition rate by overtime")
sns.barplot(data=df, x="job_satisfaction", y="attrition_flag", ax=ax[1]); ax[1].set_title("Attrition rate by job satisfaction")
sns.histplot(df.performance_score, kde=True, ax=ax[2]); ax[2].set_title("Performance score")
plt.tight_layout(); plt.show()
plt.figure(figsize=(9,7)); sns.heatmap(df[num_cols+["performance_score","attrition_flag"]].corr(), cmap="coolwarm", center=0); plt.title("Correlation heatmap"); plt.show()

## 3. Attrition prediction: Random Forest classifier

In [ ]:
def make_pipe(model, feats):
    cats = [f for f in feats if f in cat_cols]; nums = [f for f in feats if f not in cat_cols]
    return Pipeline([("prep", ColumnTransformer([("n","passthrough",nums),("c",OneHotEncoder(handle_unknown="ignore"),cats)])),
                     ("model", model)])

feats = num_cols + cat_cols + ["performance_score"]
Xtr, Xte, ytr, yte = train_test_split(df[feats], df.attrition_flag, test_size=.2, stratify=df.attrition_flag, random_state=42)
clf = make_pipe(RandomForestClassifier(n_estimators=300, min_samples_leaf=4, class_weight="balanced_subsample", random_state=42, n_jobs=-1), feats).fit(Xtr, ytr)
print(classification_report(yte, clf.predict(Xte)))
print("ROC-AUC:", round(roc_auc_score(yte, clf.predict_proba(Xte)[:,1]), 3))

## 4. Performance prediction: Random Forest regressor

In [ ]:
bfeats = num_cols + cat_cols
Xtr, Xte, ytr, yte = train_test_split(df[bfeats], df.performance_score, test_size=.2, random_state=42)
reg = make_pipe(RandomForestRegressor(n_estimators=300, min_samples_leaf=3, max_features=.5, random_state=42, n_jobs=-1), bfeats).fit(Xtr, ytr)
p = reg.predict(Xte); print("R²:", round(r2_score(yte, p),3), " MAE:", round(mean_absolute_error(yte, p),3))

## 5. Employee segmentation (K-Means) & anomaly detection (Isolation Forest)

In [ ]:
seg = ["age","monthly_income","years_at_company","performance_score","job_satisfaction","work_life_balance",
       "job_involvement","training_hours_last_year","absenteeism_days","overtime_flag"]
X = StandardScaler().fit_transform(df[seg])
for k in range(2, 8): print(k, round(silhouette_score(X, KMeans(k, n_init=10, random_state=42).fit_predict(X)), 3))
df["segment"] = KMeans(4, n_init=20, random_state=42).fit_predict(X)
print(df.groupby("segment")[seg].mean().round(2))

iso = IsolationForest(n_estimators=300, contamination=.04, random_state=42).fit(X)
df["anomaly"] = iso.predict(X) == -1
print("Anomalies:", df.anomaly.sum()); df[df.anomaly][["employee_id","name","department"] + seg].head()

## 6. Explainable AI + HR insights
The web application (`app.py`) turns these models into an HR dashboard with per-employee AI reports, occlusion-based explanations, a risk score (0–100) and recommended actions. Run `python run.py` from the project root.